In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from pathlib import Path

torch.manual_seed(756)
np.random.seed(756)

# =========================================================
# Notebook mode
# Current notebook assumed at:
#   /MFDNN_code/PM25
# =========================================================
PM25_DIR = Path.cwd().resolve()
PROJECT_ROOT = PM25_DIR.parent
METHOD_DIR = PROJECT_ROOT / "Method"
DATA_DIR = PM25_DIR / "Data"
RESULTS_DIR = PM25_DIR / "Results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

from Method.mfdnn import MFDNN, MFDNN_predict
from Method.utils import *

# =========================================================
# Data
# =========================================================
covariate_data = np.load(DATA_DIR / "covariates_10x10.npy")
covariate_data = np.transpose(covariate_data, (3, 0, 1, 2))

y = np.load(DATA_DIR / "pm25_daily_means_2022.npy")

train_indices = pd.read_csv(
    DATA_DIR / "train_indices_list.csv",
    header=None
).to_numpy()

test_indices = pd.read_csv(
    DATA_DIR / "test_indices_list.csv",
    header=None
).to_numpy()

# =========================================================
# Settings
# =========================================================
n = 300
p = 6
# frun = 50
frun = 50
epsilon = 0.01
m = 3

domain_range = [
    [np.array([40.5, -74.3]), np.array([41.5, -73.7])] for _ in range(p)
]

model_params = {
    "num_basis": [5, 5],
    "layer_sizes": [64, 64],
    "epochs": 200,
    "val_ratio": 0.25,
    "patience": 10,
    "basis_type": "bspline",
    "degree": 3,
}

lam1_values = [0, 0.001, 0.003, 0.005, 0.007, 0.01]
lam2_values = [0, 0.001, 0.01]

# =========================================================
# Hyperparameter selection
# =========================================================
def select_best_hyperparameters(
    X_train,
    y_train,
    p,
    domain_range,
    lam1_values,
    lam2_values,
    model_params,
    epsilon=0.01,
):
    mse_results = np.zeros((len(lam1_values), len(lam2_values)))
    selection_info = {}

    y_train_mean = np.mean(y_train)
    y_train_std = np.std(y_train)

    for i, lam1 in enumerate(lam1_values):
        for j, lam2 in enumerate(lam2_values):
            try:
                train_losses, val_losses, model, l21 = MFDNN(
                    p=p,
                    resp=y_train,
                    func_cov=X_train,
                    num_basis=model_params["num_basis"],
                    layer_sizes=model_params["layer_sizes"],
                    domain_range=domain_range,
                    epochs=model_params["epochs"],
                    val_ratio=model_params["val_ratio"],
                    patience=model_params["patience"],
                    lam1=lam1,
                    lam2=lam2,
                    std_resp=True,
                    basis_type=model_params["basis_type"],
                    degree=model_params["degree"],
                )

                mse_results[i, j] = (
                    min(val_losses) if len(val_losses) > 0 else np.mean(train_losses[-10:])
                )

                selected_vars = [k for k, norm in enumerate(l21) if norm > epsilon]

                selection_info[f"{i}_{j}"] = {
                    "model": model,
                    "lam1": lam1,
                    "lam2": lam2,
                    "mse": mse_results[i, j],
                    "selected_vars": selected_vars,
                    "y_mean": y_train_mean,
                    "y_std": y_train_std,
                    "l21": l21,
                }

            except Exception:
                mse_results[i, j] = np.inf
                selection_info[f"{i}_{j}"] = {
                    "model": None,
                    "lam1": lam1,
                    "lam2": lam2,
                    "mse": np.inf,
                    "selected_vars": [],
                    "y_mean": y_train_mean,
                    "y_std": y_train_std,
                    "l21": None,
                }

    best_idx = np.unravel_index(np.argmin(mse_results), mse_results.shape)
    best_candidate = selection_info[f"{best_idx[0]}_{best_idx[1]}"]

    return best_candidate["lam1"], best_candidate["lam2"], best_candidate


# =========================================================
# Evaluation
# =========================================================
def evaluate_on_test_set(best_candidate, X_test, y_test, p, domain_range, model_params):
    try:
        y_mean = best_candidate["y_mean"]
        y_std = best_candidate["y_std"]

        test_predictions_normalized = MFDNN_predict(
            p,
            best_candidate["model"],
            X_test,
            model_params["num_basis"],
            domain_range,
            basis_type=model_params["basis_type"],
            degree=model_params["degree"],
        )
        test_predictions_original = (
            test_predictions_normalized.detach().numpy() * y_std + y_mean
        )

        test_mse = np.mean((test_predictions_original.flatten() - y_test) ** 2)
        test_rmse = np.sqrt(test_mse)

        return test_rmse, test_predictions_original
    except Exception:
        return np.inf, None


# =========================================================
# Main experiment
# =========================================================
all_results = {
    "test_rmse": [],
    "rank_sequences": []
}

for run_idx in range(frun):
    if run_idx % 10 == 0:
        print(f"Run {run_idx + 1}/{frun}")

    X_train = covariate_data[:, train_indices[run_idx], :, :]
    X_test = covariate_data[:, test_indices[run_idx], :, :]

    y_train = y[train_indices[run_idx]]
    y_test = y[test_indices[run_idx]]

    lam1, lam2, best_candidate = select_best_hyperparameters(
        X_train,
        y_train,
        p=p,
        domain_range=domain_range,
        lam1_values=lam1_values,
        lam2_values=lam2_values,
        model_params=model_params,
        epsilon=epsilon,
    )

    test_rmse, _ = evaluate_on_test_set(
        best_candidate,
        X_test,
        y_test,
        p=p,
        domain_range=domain_range,
        model_params=model_params,
    )
    all_results["test_rmse"].append(test_rmse)

    l21 = best_candidate.get("l21", None)
    if l21 is not None:
        ranked_vars = [int(k) for k in np.argsort(-l21.detach().cpu().numpy())]
    else:
        ranked_vars = None

    all_results["rank_sequences"].append(ranked_vars)

# =========================================================
# Print RMSE summary
# =========================================================
rmse_mean = np.mean(all_results["test_rmse"])
rmse_std = np.std(all_results["test_rmse"])

print(f"\nTest RMSE Mean: {rmse_mean:.4f}")
print(f"Test RMSE Std: {rmse_std:.4f}")

# =========================================================
# Print top-m frequencies
# =========================================================
topm_count = np.zeros(p, dtype=int)
for seq in all_results["rank_sequences"]:
    if seq is not None:
        for var in seq[:m]:
            topm_count[var] += 1

print(f"\nTop-{m} frequency:")
for i, count in enumerate(topm_count):
    print(f"Predictor {i + 1}: {count} times")

# =========================================================
# Save all results together
# =========================================================
df_vars = pd.DataFrame({
    "Predictor": [f"X{i+1}" for i in range(p)],
    f"top{m}_count": topm_count
})

df_rmse = pd.DataFrame({
    "Predictor": ["RMSE_mean", "RMSE_std"],
    f"top{m}_count": [rmse_mean, rmse_std]
})

df_summary = pd.concat([df_vars, df_rmse], ignore_index=True)

save_path = RESULTS_DIR / f"PM25_MFDNN.csv"
df_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"\nSaved to: {save_path}")

Run 1/50
Run 11/50
Run 21/50
Run 31/50
Run 41/50

Test RMSE Mean: 2.4883
Test RMSE Std: 0.2243

Top-3 frequency:
Predictor 1: 30 times
Predictor 2: 41 times
Predictor 3: 0 times
Predictor 4: 18 times
Predictor 5: 27 times
Predictor 6: 34 times

Saved to: /Users/wangdongxue/Desktop/MFDNN_STCO_revise/MFDNN_code/PM25/Results/PM25_MFDNN.csv
